In [17]:
from pydantic import BaseModel
from typing import List, Dict


In [18]:
class Patient(BaseModel):
    name: str
    age: int
    medical_history: List[str]
    contract_details: Dict[str, str]

In [19]:
def createPatient(patient: Patient):
    print(patient.name)
    print(patient.age)
    print(patient.medical_history)
    print(patient.contract_details)
    print("Patient created successfully")


In [20]:
patient_data = {
    "name": "John Doe",
    "age": 30,
    "medical_history": ["Diabetes", "Hypertension"],
    "contract_details": {"email": "john.doe@example.com", "phone": "123-456-7890"}
}
patient = Patient(**patient_data)

In [21]:
patient

Patient(name='John Doe', age=30, medical_history=['Diabetes', 'Hypertension'], contract_details={'email': 'john.doe@example.com', 'phone': '123-456-7890'})

In [22]:
patient_data

{'name': 'John Doe',
 'age': 30,
 'medical_history': ['Diabetes', 'Hypertension'],
 'contract_details': {'email': 'john.doe@example.com',
  'phone': '123-456-7890'}}

In [24]:
createPatient(patient)

John Doe
30
['Diabetes', 'Hypertension']
{'email': 'john.doe@example.com', 'phone': '123-456-7890'}
Patient created successfully


In [25]:
from typing import Optional

In [27]:
#Optinal fields and default values
class User(BaseModel):
    name: str
    age: int
    email: Optional[str] = None
    contactDetails: Dict[str, str]
    favoriteColors: List[str]



In [29]:
def userInfo(user: User):
    print("User Information:")
    print(user.name)
    print(user.age)
    print(user.email)
    print(user.contactDetails)
    print(user.favoriteColors)

In [30]:
userData = {
    "name": "Alice Smith",
    "age": 25,
    "contactDetails": {"Phone": "123-456-7890"},
    "favoriteColors": ["blue", "green"]
}

In [32]:
user = User(**userData)
userInfo(user)

User Information:
Alice Smith
25
None
{'Phone': '123-456-7890'}
['blue', 'green']


In [33]:
from pydantic import BaseModel, Field
from uuid import uuid4
import re

class User(BaseModel):
    # Required field with description, title, and examples
    id: str = Field(
        default_factory=lambda: str(uuid4()),   # dynamic default
        title="User Identifier",
        description="A unique identifier for the user",
        examples=["123e4567-e89b-12d3-a456-426614174000"]
    )

    # String with constraints
    username: str = Field(
        ...,
        min_length=3,
        max_length=20,
        pattern=r"^[a-zA-Z0-9_]+$",   # regex constraint
        description="Username must be alphanumeric with underscores",
        examples=["john_doe", "alice123"]
    )

    # Numeric field with constraints
    age: int = Field(
        ...,
        ge=0,   # greater or equal to 0
        le=120, # less or equal to 120
        description="Age must be between 0 and 120",
        examples=[25, 40]
    )

    # Float with strict typing
    balance: float = Field(
        default=0.0,
        ge=0.0,
        strict=True,   # disallow coercion from str → float
        description="Account balance in USD",
        examples=[100.50, 0.0]
    )

    # Alias usage
    email: str = Field(
        ...,
        alias="user_email",   # JSON key expected as "user_email"
        description="Primary email address of the user",
        examples=["user@example.com"]
    )


In [34]:
user_data = {
    "username": "john_doe",
    "age": 30,
    "balance": 150.75,
    "user_email": "john.doe@example.com"
}


In [39]:
from pydantic import BaseModel, Field, field_validator
from typing import Optional
import re

class User(BaseModel):
    username: str = Field(..., min_length=3, max_length=20)
    age: int = Field(..., ge=0, le=120)
    email: Optional[str] = Field(None)
    balance: float = Field(default=0.0, ge=0.0)

    # Simple validation: enforce lowercase usernames
    @field_validator("username")
    @classmethod
    def username_must_be_lowercase(cls, v):
        if not v.islower():
            raise ValueError("Username must be lowercase")
        return v

    # Regex validation: email format check
    @field_validator("email")
    @classmethod
    def email_must_be_valid(cls, v):
        if v is None:
            return v
        pattern = r"^[\w\.-]+@[\w\.-]+\.\w+$"
        if not re.match(pattern, v):
            raise ValueError("Invalid email format")
        return v

    # Multiple fields validated together (pre=True)
    @field_validator("balance", mode="before")
    @classmethod
    def balance_as_string_allowed(cls, v):
        # Convert string to float if possible
        if isinstance(v, str):
            try:
                return float(v)
            except ValueError:
                raise ValueError("Balance must be a number")
        return v

    # Complex logic: age restriction
    @field_validator("age")
    @classmethod
    def age_must_be_adult(cls, v):
        if v < 18:
            raise ValueError("User must be at least 18 years old")
        return v


In [ ]:
# Valid input
user = User(username="john_doe", age=25, email="john@example.com", balance="100.5")
print(user)

# Invalid input (username not lowercase)
User(username="johnDoe", age=15, email="john@example.com")
# ValidationError: Username must be lowercase

# Invalid input (age < 18)
User(username="john_doe", age=15, email="john@example.com")
# ValidationError: User must be at least 18 years old


username='john_doe' age=25 email='john@example.com' balance=100.5


ValidationError: 2 validation errors for User
username
  Value error, Username must be lowercase [type=value_error, input_value='johnDoe', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error
age
  Value error, User must be at least 18 years old [type=value_error, input_value=15, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error

: 

In [ ]:
from pydantic import comput